# YOLOv8 Training on Colab (T4 GPU)

This notebook trains YOLOv8 on your PlantVillage object-detection dataset.

## 1) Runtime Setup
- In Colab: `Runtime` -> `Change runtime type` -> `T4 GPU`
- Then run cells top to bottom.

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
!pip -q install --upgrade ultralytics pyyaml
import ultralytics
print('Ultralytics version:', ultralytics.__version__)

## 2) Mount Drive and Unzip Dataset

Upload a zip of your prepared dataset folder to Drive, for example:
`/content/drive/MyDrive/PlantVillage_for_object_detection.zip`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import zipfile
from pathlib import Path

ZIP_PATH = '/content/drive/MyDrive/PlantVillage_for_object_detection.zip'  # change if needed
EXTRACT_ROOT = Path('/content/datasets')
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_ROOT)

print('Extracted to:', EXTRACT_ROOT)

In [ ]:
# If your zip contains PlantVillage_for_object_detection/Dataset/
DATASET_DIR = EXTRACT_ROOT / 'PlantVillage_for_object_detection' / 'Dataset'
print('DATASET_DIR exists:', DATASET_DIR.exists())
print('Train images dir exists:', (DATASET_DIR / 'train' / 'images').exists())

## 3) Write data.yaml for Colab paths

In [ ]:
import yaml

names = [
    'Apple___Apple_scab',
    'Apple___Black_rot',
    'Apple___Cedar_apple_rust',
    'Apple___healthy',
    'Blueberry___healthy',
    'Cherry___Powdery_mildew',
    'Cherry___healthy',
    'Corn___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn___Common_rust',
    'Corn___Northern_Leaf_Blight',
    'Corn___healthy',
    'Grape___Black_rot',
    'Grape___Esca_(Black_Measles)',
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
    'Grape___healthy',
    'Orange___Haunglongbing_(Citrus_greening)',
    'Peach___Bacterial_spot',
    'Peach___healthy',
    'Pepper,_bell___Bacterial_spot',
    'Pepper,_bell___healthy',
    'Potato___Early_blight',
    'Potato___Late_blight',
    'Potato___healthy',
    'Raspberry___healthy',
    'Soybean___healthy',
    'Squash___Powdery_mildew',
    'Strawberry___Leaf_scorch',
    'Strawberry___healthy',
    'Tomato___Bacterial_spot',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy'
]

data = {
    'path': str(DATASET_DIR),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': len(names),
    'names': {i: n for i, n in enumerate(names)}
}

yaml_path = DATASET_DIR / 'data_colab.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=False)

print('Saved:', yaml_path)
print(open(yaml_path, 'r', encoding='utf-8').read())

## 4) Quick Dataset Sanity Check

In [ ]:
from pathlib import Path

def count_files(d, ext):
    return len(list(Path(d).glob(ext)))

for split in ['train', 'val', 'test']:
    img_count = count_files(DATASET_DIR / split / 'images', '*.jpg')
    lbl_count = count_files(DATASET_DIR / split / 'labels', '*.txt')
    print(split, 'images:', img_count, 'labels:', lbl_count)

## 5) Train YOLOv8

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Rebuild YAML path defensively in case runtime restarted.
yaml_path = (DATASET_DIR / 'data_colab.yaml').resolve()
assert yaml_path.exists(), f'Missing YAML: {yaml_path}. Run dataset + YAML cells first.'

print('Training YAML:', yaml_path)

model = YOLO('yolov8s.pt')  # use yolov8n.pt for faster training

results = model.train(
    data=str(yaml_path),
    epochs=80,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project='runs_colab',
    name='plantvillage_yolov8s_t4',
    pretrained=True,
    optimizer='auto',
    cos_lr=True,
    patience=20,
    seed=42,
    amp=True,
    cache=False,
    plots=True
)

## 6) Validate and Test Inference

In [ ]:
best_model = YOLO('runs_colab/plantvillage_yolov8s_t4/weights/best.pt')
metrics = best_model.val(data=str(yaml_path), split='val')
print(metrics)

In [ ]:
test_images = str(DATASET_DIR / 'test' / 'images')
pred = best_model.predict(source=test_images, imgsz=640, conf=0.25, save=True)
print('Predictions saved in runs/detect/...')

## 7) Export and Save to Drive

In [ ]:
best_model.export(format='onnx')
best_model.export(format='torchscript')

In [ ]:
!cp -r runs_colab /content/drive/MyDrive/
print('Copied runs_colab to Google Drive')